# Chạy server (host_server): tải embedding + map, build FAISS, chạy API

Bật **Internet** và **GPU**. Gắn `rclone.conf` tại đường dẫn bên dưới hoặc tạo
Kaggle Secret `RCLONE_CONFIG` chứa nội dung file. Tạo Secret `NGROK_AUTHTOKEN`.
Điền `embedding`, `map` và `range` (hai đầu đều được lấy) trong từng dictionary của `DATASETS`. V3C và LHE được tải song song, sau đó build index riêng lần lượt để hạn chế RAM. Có thể bỏ một entry nếu chỉ muốn host một dataset. Notebook clone nhánh `vbs-dev` và gọi pipeline có sẵn của repo; nhánh remote cần có cấu hình của cả hai dataset.

Dữ liệu sau giải nén nằm trong `/kaggle/working/event-retrieval-system/data`, với
cùng cây thư mục như local. API và vLLM chạy trên Kaggle sau khi index xong.
Kaggle chỉ giữ server hoạt động khi phiên notebook còn chạy.


In [ ]:
# 1. Cấu hình và clone repo
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/iamtro-2006/event-retrieval-system.git"
REPO_BRANCH = "vbs-dev"
REPO_DIR = Path("/kaggle/working/event-retrieval-system")
BACKEND_DIR = REPO_DIR / "backend"
# Mỗi dataset có folder Drive và range video riêng (inclusive).
DATASETS = {
    "V3C": {
        "embedding": "1gLyt3E6w6AdNh6YKxO5Zqn_Nj6zlwFfq",
        "map": "1N4dWx4Od2bgazMf6_PEj0TKDR5t2LneD",
        "range": (1, 100),
    },
    "LHE": {
        "embedding": "",  # điền folder ID Drive của LHE
        "map": "",
        "range": (1, 75),
    },
}
DEFAULT_DATASET = "V3C"
if not DATASETS or DEFAULT_DATASET not in DATASETS:
    raise ValueError("DEFAULT_DATASET phải nằm trong DATASETS")
for dataset, spec in DATASETS.items():
    if dataset not in {"V3C", "LHE"}:
        raise ValueError(f"Dataset không hợp lệ: {dataset}")
    start, end = spec["range"]
    if not (isinstance(start, int) and isinstance(end, int)
            and 1 <= start <= end <= (99 if dataset == "LHE" else 99999)):
        raise ValueError(f"Range video không hợp lệ cho {dataset}: {spec['range']}")
    if not spec["embedding"] or not spec["map"]:
        raise ValueError(f"Thiếu Drive folder ID embedding/map cho {dataset}")

def video_name(dataset, number):
    return f"LHE{number:02d}" if dataset == "LHE" else f"{number:05d}"
RCLONE_CONFIG_SOURCE = Path("/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf")
WORK_DIR = Path("/kaggle/tmp/retrieval_server")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO_DIR)], check=True)
assert (BACKEND_DIR / "scripts/retrieval/run.py").is_file(), "Repo clone chưa hoàn chỉnh"

import yaml
app_path = BACKEND_DIR / "configs/app.yaml"
app_cfg = yaml.safe_load(app_path.read_text(encoding="utf-8"))
index_cfg = yaml.safe_load((BACKEND_DIR / "configs/indexing.yaml").read_text(encoding="utf-8"))
built_indexes = {}
for dataset in DATASETS:
    dataset_cfg = app_cfg["datasets"]["available"].get(dataset.lower())
    assert dataset_cfg is not None, f"Nhánh vbs-dev chưa có config {dataset}; hãy đẩy thay đổi local lên remote."
    base_index = dataset_cfg["semantic_models"]["qwen3-vl-embedding-2b"]["index_dir"]
    built_indexes[dataset] = base_index if base_index.endswith("_vllm") else base_index + "_vllm"
    assert index_cfg["datasets"]["available"][dataset.lower()]["index"]["output_dir"] == built_indexes[dataset]
app_cfg["datasets"]["available"] = {
    dataset.lower(): app_cfg["datasets"]["available"][dataset.lower()] for dataset in DATASETS
}
app_cfg["datasets"]["default"] = DEFAULT_DATASET.lower()
assert app_cfg["semantic"]["models"][0]["model_extra"]["engine"] == "vllm"
app_path.write_text(yaml.safe_dump(app_cfg, sort_keys=False), encoding="utf-8")
print("Repo/config đã sẵn sàng:", list(DATASETS), REPO_DIR)


In [ ]:
# 2. Cài dependency cho index/API và rclone; chuẩn bị config Drive
import sys
import urllib.request
import zipfile

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "fastapi",
                "faster-whisper", "httpx", "numba", "pandas", "python-multipart", "PyYAML",
                "requests", "Pillow", "elasticsearch", "tqdm", "uvicorn", "pyngrok", "uv",
                "opencv-python-headless", "ffmpeg-python", "scipy", "scikit-learn",
                "python-dotenv", "openai"], check=True)

rclone = shutil.which("rclone")
if not rclone:
    release = "v1.75.2"
    archive = WORK_DIR / f"rclone-{release}-linux-amd64.zip"
    url = f"https://github.com/rclone/rclone/releases/download/{release}/{archive.name}"
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as zf:
        member = next(m for m in zf.namelist() if m.endswith("/rclone"))
        rclone = WORK_DIR / "rclone"
        rclone.write_bytes(zf.read(member))
        rclone.chmod(0o755)
    rclone = str(rclone)

rclone_config = WORK_DIR / "rclone.conf"
config_secret = secret("RCLONE_CONFIG")
if config_secret:
    rclone_config.write_text(config_secret, encoding="utf-8")
elif RCLONE_CONFIG_SOURCE.is_file():
    shutil.copy2(RCLONE_CONFIG_SOURCE, rclone_config)
else:
    raise FileNotFoundError("Gắn rclone.conf hoặc tạo Kaggle Secret RCLONE_CONFIG")
rclone_config.chmod(0o600)
subprocess.run([rclone, "--config", str(rclone_config), "listremotes"], check=True)


In [ ]:
# 3. Tải song song hai dataset; thư mục ZIP và dữ liệu tách riêng
from concurrent.futures import ThreadPoolExecutor

def drive_files(folder_id):
    result = subprocess.run([rclone, "--config", str(rclone_config),
                             "--drive-root-folder-id", folder_id, "lsf", "iamtro:",
                             "--files-only", "--format", "p"], check=True,
                            capture_output=True, text=True)
    return set(result.stdout.splitlines())

def download_dataset(dataset, spec):
    EMBED_DIR = REPO_DIR / "data/embeddings/embeddings_qwen3_vl_embedding_2b_vllm/" / dataset
    MAP_DIR = REPO_DIR / "data/map_keyframes" / dataset
    ZIP_DIR = WORK_DIR / "embedding_zips" / dataset
    EMBED_DIR.mkdir(parents=True, exist_ok=True)
    MAP_DIR.mkdir(parents=True, exist_ok=True)
    ZIP_DIR.mkdir(parents=True, exist_ok=True)
    (REPO_DIR / "data/keyframes" / dataset).mkdir(parents=True, exist_ok=True)
    (REPO_DIR / "data/videos" / dataset).mkdir(parents=True, exist_ok=True)

    start, end = spec["range"]
    ids = [video_name(dataset, i) for i in range(start, end + 1)]
    embedding_files = drive_files(spec["embedding"])
    map_files = drive_files(spec["map"])
    missing_zips = [name + ".zip" for name in ids if name + ".zip" not in embedding_files]
    if missing_zips:
        print(f"Bỏ qua {len(missing_zips)} ZIP chưa có trên Drive:", missing_zips)
    available_ids = [video_id for video_id in ids if video_id + ".zip" in embedding_files
                     or list((EMBED_DIR / video_id).glob("*.npy"))]
    if not available_ids:
        raise RuntimeError(f"Chưa có embedding {dataset} trong range đã chọn để build index")
    missing_maps = [name + ".csv" for name in ids if name + ".csv" not in map_files]
    if missing_maps:
        print("Thiếu map CSV (index vẫn build, timestamp các video này có thể thiếu):", missing_maps)

    for video_id in available_ids:
        target = EMBED_DIR / video_id
        if not list(target.glob("*.npy")):
            zip_path = ZIP_DIR / f"{video_id}.zip"
            subprocess.run([rclone, "--config", str(rclone_config),
                            "--drive-root-folder-id", spec["embedding"], "copyto",
                            f"iamtro:{video_id}.zip", str(zip_path), "-P"], check=True)
            with zipfile.ZipFile(zip_path) as zf:
                members = [m for m in zf.infolist() if not m.is_dir() and m.filename.endswith(".npy")]
                if not members:
                    raise ValueError(f"ZIP rỗng: {zip_path}")
                target.mkdir(parents=True, exist_ok=True)
                for member in members:
                    # ZIP có thể chứa <video_id>/<frame>.npy hoặc chỉ <frame>.npy.
                    # Chỉ nhận tên frame, tránh path traversal và thư mục lồng đôi.
                    frame_name = Path(member.filename).name
                    with zf.open(member) as source, (target / frame_name).open("wb") as dest:
                        shutil.copyfileobj(source, dest)
            zip_path.unlink()  # giữ dữ liệu đã giải nén, bỏ ZIP để tiết kiệm dung lượng
        csv_path = MAP_DIR / f"{video_id}.csv"
        if f"{video_id}.csv" in map_files and not csv_path.is_file():
            subprocess.run([rclone, "--config", str(rclone_config),
                            "--drive-root-folder-id", spec["map"], "copyto",
                            f"iamtro:{video_id}.csv", str(csv_path), "-P"], check=True)
        print(dataset, video_id, len(list(target.glob("*.npy"))), "frames",
              "map" if csv_path.is_file() else "no map")
    print("Embedding:", EMBED_DIR)
    print("Map:", MAP_DIR)
    return available_ids

with ThreadPoolExecutor(max_workers=len(DATASETS)) as pool:
    downloads = {dataset: pool.submit(download_dataset, dataset, spec)
                 for dataset, spec in DATASETS.items()}
    available_ids_by_dataset = {dataset: future.result() for dataset, future in downloads.items()}


In [ ]:
# 4. Build FAISS + vector cache riêng cho từng dataset, lần lượt để hạn chế RAM
import faiss
import pandas as pd

for dataset in DATASETS:
    subprocess.run([sys.executable, "-m", "scripts.retrieval.run", "--task", "build-index",
                    "--dataset", dataset.lower()], cwd=BACKEND_DIR, check=True)

    INDEX_DIR = REPO_DIR / built_indexes[dataset].removeprefix("../")
    index = faiss.read_index(str(INDEX_DIR / "keyframes.faiss"))
    metadata = pd.read_csv(INDEX_DIR / "metadata.csv", dtype={"video_id": str})
    assert index.ntotal == len(metadata) > 0
    assert set(metadata["dataset"]) == {dataset}, "Index trộn source group khác"
    assert set(metadata["collection"]) == {dataset.lower()}, "Index trộn dataset khác"
    assert set(metadata["video_id"]).issubset(set(available_ids_by_dataset[dataset])), "video_id trong metadata không khớp dữ liệu đã tải"
    print(f"{dataset}: đã index {index.ntotal:,} keyframes từ {metadata.video_id.nunique()} video")

    CACHE_PATH = INDEX_DIR / "vectors_fp32.npy"
    subprocess.run([sys.executable, "-m", "scripts.retrieval.run", "--task", "build-vector-cache",
                    "--config", "configs/app.yaml", "--dataset", dataset.lower(), "--output", str(CACHE_PATH),
                    "--dtype", "float32"], cwd=BACKEND_DIR, check=True)
    cache = __import__("numpy").load(CACHE_PATH, mmap_mode="r")
    assert cache.shape == (index.ntotal, index.d)
    del cache

    print("Vector cache sẵn sàng:", dataset, CACHE_PATH)
    del index, metadata

# Để backend dùng cache vừa tạo thay vì tái dựng toàn bộ vector vào RAM.
app_path = BACKEND_DIR / "configs/app.yaml"
app_cfg["semantic"]["models"][0]["vector_cache_mode"] = "memmap"
app_path.write_text(yaml.safe_dump(app_cfg, sort_keys=False), encoding="utf-8")


In [ ]:
# 5. Cài và khởi động vLLM riêng; API dùng code của repo
import json
import time
import urllib.error

VLLM_VENV = WORK_DIR / "vllm-venv"
if not (VLLM_VENV / "bin/vllm").exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(VLLM_VENV)], check=True)
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", str(VLLM_VENV / "bin/python"),
                    "--torch-backend=auto", "-r", str(BACKEND_DIR / "requirements-vllm.txt")], check=True)

def wait_http(url, seconds, ready):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                payload = json.load(response)
            if ready(payload):
                return payload
        except (OSError, ValueError, urllib.error.URLError):
            pass
        time.sleep(3)
    raise TimeoutError(f"Chưa sẵn sàng: {url}")

vllm_log = WORK_DIR / "vllm.log"
vllm_handle = vllm_log.open("a", encoding="utf-8")
vllm_process = subprocess.Popen([str(VLLM_VENV / "bin/vllm"), "serve",
                                 "Qwen/Qwen3-VL-Embedding-2B", "--runner", "pooling",
                                 "--max-model-len", "8192", "--host", "127.0.0.1",
                                 "--port", "8001"], stdout=vllm_handle, stderr=subprocess.STDOUT)
try:
    wait_http("http://127.0.0.1:8001/v1/models", 600,
              lambda payload: any("Qwen3-VL-Embedding-2B" in item["id"] for item in payload.get("data", [])))
except Exception:
    print(vllm_log.read_text(encoding="utf-8")[-5000:])
    raise

api_log = WORK_DIR / "api.log"
api_handle = api_log.open("a", encoding="utf-8")
api_env = os.environ.copy()
api_env.update(QWEN_EMBEDDING_ENGINE="vllm", QWEN_VLLM_URL="http://127.0.0.1:8001/v1")
api_process = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.api.main:app",
                                "--host", "127.0.0.1", "--port", "8000"],
                               cwd=BACKEND_DIR, env=api_env, stdout=api_handle,
                               stderr=subprocess.STDOUT)
try:
    # Kiểm tra từng dataset, không chỉ trạng thái của dataset mặc định.
    for dataset in DATASETS:
        wait_http(f"http://127.0.0.1:8000/api/search/models?dataset={dataset.lower()}", 180,
                  lambda payload: bool(payload.get("models")))
    health = wait_http("http://127.0.0.1:8000/", 180,
                       lambda payload: payload.get("retrieval_system_ready") is True)
except Exception:
    print(api_log.read_text(encoding="utf-8")[-5000:])
    raise
print("API health:", health)


In [ ]:
# 6. Mở tunnel ngrok cho frontend
from pyngrok import ngrok

ngrok_token = secret("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("Thiếu Kaggle Secret NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, "http")
print("API URL:", tunnel.public_url)
print("Frontend VITE_API_BASE_URL=" + tunnel.public_url)
print("Kaggle cần giữ phiên notebook hoạt động để server tiếp tục phục vụ.")